In [ ]:
!apt-get update -qq
!apt-get install -y -qq wget unzip chromium-chromedriver
!pip install -q -U selenium pandas beautifulsoup4 requests
!pip install  -q PyPDF2
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb
!pip install transformers torch pandas matplotlib seaborn

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import requests
import re
import time
import random
from io import BytesIO
import PyPDF2
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.decomposition import PCA
from collections import Counter
from nltk import bigrams

In [ ]:
import subprocess
chrome_version = subprocess.check_output(['/usr/bin/google-chrome-stable', '--version']).decode().strip()
print(f"Chrome: {chrome_version}")

In [ ]:
options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.binary_location = "/usr/bin/google-chrome-stable"

driver = webdriver.Chrome(options=options)
driver.get("https://www.google.com")
driver.quit()

In [ ]:
def make_colab_driver(headless=True):
    options = Options()

    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-infobars")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    options.add_argument("--lang=en-GB")
    options.binary_location = "/usr/bin/google-chrome-stable"

    driver = webdriver.Chrome(options=options)

    try:
        driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": """
                Object.defineProperty(navigator, 'webdriver', {
                    get: () => undefined
                });
            """
        })
    except:
        pass

    return driver

In [ ]:
class BOEScraper:
    def __init__(self, headless=True):
        self.driver = make_colab_driver(headless=headless)
        self.wait = WebDriverWait(self.driver, 20)

        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        })

        self.results = []
        self.seen_urls = set()

    def scrape_cbdc_articles(self):
        base_url = "https://www.bankofengland.co.uk/news"

        self.driver.get("https://www.bankofengland.co.uk")
        time.sleep(2)

        try:
            cookie_btn = self.driver.find_element(By.XPATH, "//button[contains(text(), 'Accept')]")
            cookie_btn.click()
            time.sleep(1)
        except:
            pass

        self.driver.get(base_url)
        time.sleep(3)

        if not self._search_cbdc():
            print("fail")
            self.driver.quit()
            return pd.DataFrame()
        time.sleep(4)


        all_articles = []

        for page_num in range(1, 4):

            try:
                self.wait.until(EC.presence_of_element_located((By.TAG_NAME, "main")))
                time.sleep(2)
            except:
                print(f"Main content not found on page {page_num}")

            self._scroll_page_thoroughly()
            time.sleep(2)

            try:

                self.wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='/']")))
            except:
                pass

            soup = BeautifulSoup(self.driver.page_source, 'html.parser')
            page_articles = self._extract_all_links(soup)
            print(f"Found {len(page_articles)} articles on page {page_num}")

            before_count = len(all_articles)
            for article in page_articles:
                if article not in all_articles:
                    all_articles.append(article)

            new_count = len(all_articles) - before_count
            print(f"Added {new_count} new unique articles (total: {len(all_articles)})")

            if page_num < 3:
                if self._click_next_page_button():
                    time.sleep(4)
                else:
                    print(f"Could not move to page {page_num + 1}")
                    break

        print(f"Total unique articles: {len(all_articles)}")

        if len(all_articles) == 0:
            print("No articles found")
            self.driver.quit()
            return pd.DataFrame()

        for i, (url, title) in enumerate(all_articles, start=1):
            if i % 10 == 0:
                print(f"   Progress: {i}/{len(all_articles)}")
            self.scrape_article(url, title)
            time.sleep(random.uniform(0.3, 0.7))

        print(f"{len(self.results)} articles scraped")
        self.driver.quit()
        return pd.DataFrame(self.results)

    def _search_cbdc(self):
        try:
            search_box = self.driver.find_element(By.XPATH, "//input[@placeholder='search this area...']")
            search_box.clear()
            time.sleep(0.5)
            search_box.send_keys("cbdc")
            time.sleep(0.5)
            search_box.send_keys(Keys.RETURN)
            print("Searched for 'cbdc'")
            return True
        except Exception as e:
            print(f"Search failed: {e}")
            return False

    def _scroll_page_thoroughly(self):

        for i in range(5):
            self.driver.execute_script(f"window.scrollTo(0, {(i+1) * 300});")
            time.sleep(0.3)

        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)

        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight - 500);")
        time.sleep(0.5)
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)

    def _click_next_page_button(self):

        try:
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1)

            next_selectors = [
                "//a[@aria-label='Next page']",
                "//button[@aria-label='Next page']",
                "//a[contains(@class, 'pagination') and contains(., '›')]",
                "//button[contains(@class, 'pagination') and contains(., '›')]",
                "//a[contains(@class, 'next')]",
                "//button[contains(@class, 'next')]",
            ]

            for selector in next_selectors:
                try:
                    btn = self.driver.find_element(By.XPATH, selector)
                    if btn.is_displayed() and btn.is_enabled():
                        self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                        time.sleep(0.5)

                        try:
                            btn.click()
                            return True

                        except:
                            self.driver.execute_script("arguments[0].click();", btn)
                            return True
                except:
                    continue

            try:
                active_page = self.driver.find_element(By.XPATH, "//a[contains(@class, 'active') or @aria-current='page']")
                current_num = int(active_page.text.strip())
                next_num = current_num + 1

                # next page number
                next_page_btn = self.driver.find_element(By.XPATH, f"//a[text()='{next_num}' and not(contains(@class, 'active'))]")
                self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_page_btn)
                time.sleep(0.5)
                next_page_btn.click()
                return True
            except:
                pass

            return False

        except Exception as e:
            print(f"Pagination error: {e}")
            return False

    def _extract_all_links(self, soup):

        articles = []
        seen = set()

        all_links = soup.find_all('a', href=True)

        for a in all_links:
            href = a.get('href', '').strip()
            if not href:
                continue

            if href.startswith('/'):
                href = 'https://www.bankofengland.co.uk' + href
            elif not href.startswith('http'):
                continue

            if self._is_boe_article(href):
                if href not in seen:
                    seen.add(href)
                    title = a.get_text(strip=True) or "Untitled"
                    articles.append((href, title))

        return articles

    def _is_boe_article(self, url):
        url_lower = url.lower()
        if not url_lower.startswith('https://www.bankofengland.co.uk'):
            return False

        if any(x in url_lower for x in ['?query', '?search', '/news?', '#']):
            return False

        patterns = ['/minutes/', '/speech/', '/working-paper/', '/publication/', '/article/',
                   '/quarterly-bulletin/', '/report/', '/news/', '/events/', '/paper/']

        return any(p in url_lower for p in patterns)

    def scrape_article(self, url, fallback_title):
        if url in self.seen_urls:
            return
        self.seen_urls.add(url)

        try:
            resp = self.session.get(url, timeout=20)
            soup = BeautifulSoup(resp.content, 'html.parser')

            h1 = soup.find('h1')
            title = h1.get_text(strip=True) if h1 else fallback_title
            date_str, year = self._extract_published_date(soup, url)
            main = (soup.find('main') or soup.find('article') or
                   soup.find('div', class_=re.compile(r'content|article', re.I)))

            if main:
                for tag in main.find_all(['nav', 'aside', 'footer', 'script', 'style']):
                    tag.decompose()
                text = ' '.join(main.get_text(' ', strip=True).split())
            else:
                text = ' '.join(soup.get_text(' ', strip=True).split())

            article_type = self._get_type(url)

            record = {
                'title': title,
                'date': date_str,
                'year': year,
                'type': article_type,
                'url': url,
                'source': 'HTML',
                'word_count': len(text.split()),
                'content_preview': text[:500],
                'content_full': text
            }
            self.results.append(record)

            pdf_links = self._find_pdf_links(soup)
            if len(pdf_links) > 0:
                print(f"Found {len(pdf_links)} PDF(s)")

            for pdf_url in pdf_links:
                if pdf_url not in self.seen_urls:
                    self.seen_urls.add(pdf_url)
                    self._scrape_pdf(pdf_url, title, year, article_type, url)

        except Exception as e:
            pass

    def _extract_published_date(self, soup, url):
        page_text = soup.get_text()

        pub_match = re.search(r'Published on (\d{1,2} \w+ \d{4})', page_text)
        if pub_match:
            date_str = pub_match.group(1)
            year_match = re.search(r'(\d{4})', date_str)
            if year_match:
                return date_str, int(year_match.group(1))

        url_year_match = re.search(r'/(\d{4})/', url)
        if url_year_match:
            year = int(url_year_match.group(1))
            return f"Year {year}", year

        time_elem = soup.find('time', datetime=True)
        if time_elem:
            datetime_val = time_elem.get('datetime', '')
            year_match = re.match(r'(\d{4})', datetime_val)
            if year_match:
                return time_elem.get_text(strip=True), int(year_match.group(1))

        for meta_name in ['article:published_time', 'date', 'DC.date']:
            meta = soup.find('meta', attrs={'property': meta_name}) or soup.find('meta', attrs={'name': meta_name})
            if meta and meta.get('content'):
                year_match = re.search(r'(\d{4})', meta['content'])
                if year_match:
                    return meta['content'], int(year_match.group(1))

        year_match = re.search(r'\b(20\d{2})\b', page_text[:1000])
        if year_match:
            year = int(year_match.group(1))
            return f"Year {year}", year

        return "Date not found", None

    def _find_pdf_links(self, soup):
        pdf_links = []

        for a in soup.find_all('a', class_=re.compile(r'btn-pubs|btn.*pdf', re.I)):
            href = a.get('href', '')
            if href:
                if href.startswith('/'):
                    href = 'https://www.bankofengland.co.uk' + href
                if href.lower().endswith('.pdf') or '/files/' in href.lower():
                    if href not in pdf_links:
                        pdf_links.append(href)

        for a in soup.find_all('a', href=True):
            href = a['href']
            if href.startswith('/'):
                href = 'https://www.bankofengland.co.uk' + href
            if href.lower().endswith('.pdf'):
                if href not in pdf_links:
                    pdf_links.append(href)

        for a in soup.find_all('a', href=True):
            text = a.get_text(strip=True).lower()
            if 'pdf' in text or 'download' in text:
                href = a['href']
                if href.startswith('/'):
                    href = 'https://www.bankofengland.co.uk' + href
                if href.lower().endswith('.pdf') or '/-/media/boe/files/' in href.lower():
                    if href not in pdf_links:
                        pdf_links.append(href)

        return pdf_links

    def _scrape_pdf(self, pdf_url, title, year, article_type, parent_url):
        """Scrape PDF with better error handling"""
        try:
            print(f"PDF: {pdf_url}")

            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                'Referer': parent_url
            }

            resp = self.session.get(pdf_url, timeout=30, headers=headers)
            resp.raise_for_status()

            content_type = resp.headers.get('content-type', '').lower()
            if 'pdf' not in content_type and not pdf_url.lower().endswith('.pdf'):
                print(f"Not a PDF: {content_type}")
                return

            reader = PyPDF2.PdfReader(BytesIO(resp.content))

            text_parts = []
            num_pages = min(len(reader.pages), 200)

            for i, page in enumerate(reader.pages[:num_pages]):
                try:
                    text = page.extract_text() or ""
                    text_parts.append(text)
                except Exception as e:
                    print(f"Error reading page {i+1}: {e}")
                    continue

            full_text = ' '.join(' '.join(text_parts).split())

            if len(full_text.strip()) < 50:
                print(f"Cannot extract")
                return

            if year is None:
                year_match = re.search(r'\b(20\d{2})\b', full_text[:2000])
                if year_match:
                    year = int(year_match.group(1))

            record = {
                'title': f"{title} (PDF)",
                'date': f"Year {year}" if year else "Date not found",
                'year': year,
                'type': article_type,
                'url': pdf_url,
                'source': 'PDF',
                'word_count': len(full_text.split()),
                'content_preview': full_text[:500],
                'content_full': full_text
            }

            self.results.append(record)
            print(f"PDF extracted: {len(full_text.split())} words")

        except requests.exceptions.RequestException as e:
            print(f"HTTP error fetching PDF: {e}")
        except Exception as e:
            print(f"PDF extraction error: {e}")

    def _get_type(self, url):
        """Get article type"""
        url_lower = url.lower()
        if '/minutes/' in url_lower:
            return 'Minutes'
        elif '/speech/' in url_lower:
            return 'Speech'
        elif '/working-paper/' in url_lower:
            return 'Working Paper'
        elif '/publication/' in url_lower:
            return 'Publication'
        elif '/news/' in url_lower:
            return 'News'
        elif '/events/' in url_lower:
            return 'Event'
        elif '/article/' in url_lower:
            return 'Article'
        return 'Other'

In [ ]:
import subprocess
paths = ['chromium-browser', 'chromium', 'google-chrome-stable', 'google-chrome']
for p in paths:
    result = subprocess.run(['which', p], capture_output=True, text=True)
    if result.stdout.strip():
        print(f"Found: {p} -> {result.stdout.strip()}")

In [ ]:
def save_csv(df, path='boe_cbdc_articles.csv'):
    if df is None or df.empty:
        return None

    df = df.copy()
    df = df.sort_values(['year', 'source'], ascending=[False, True], na_position='last')

    cols = ['title', 'date', 'year', 'type', 'source', 'url', 'word_count', 'content_preview', 'content_full']
    df = df[cols]

    df.to_csv(path, index=False, encoding='utf-8')

    return df


def main():
    ""Run scraper""
    scraper = BOEScraper(headless=True)
    df = scraper.scrape_cbdc_articles()

    if df.empty:
        print("No articles")
        return None

    df_out = save_csv(df, 'boe_cbdc_articles.csv')

    return df_out

if __name__ == "__main__":
    df = main()

In [ ]:
print(f"Total columns: {len(df.columns)}")
print(f"Columns: {list(df.columns)}")

In [ ]:
print(df['source'].value_counts())

In [ ]:
print(df['year'].value_counts().sort_index())

In [ ]:
print(df['type'].value_counts())

In [ ]:
# check 1: count missing values in 'year'
missing_year_count = df['year'].isnull().sum()
print(f"Missing years= {missing_year_count}")
print(f"\nRows with missing year:")
print(df[df['year'].isnull()][['title', 'year', 'url', 'word_count']])

In [ ]:
print(f"Before removal: {len(df)} rows")
df = df.dropna(subset=['year'])
print(f"After removal: {len(df)} rows")
print(f"Rows removed: {missing_year_count}")
print(f"Missing years after removal: {df['year'].isnull().sum()}")
df.to_csv('boe_cbdc_articles.csv', index=False, encoding='utf-8')

In [ ]:
# check 2: Duplicate URLs
duplicate_url_count = df['url'].duplicated().sum()
print(f"Duplicate URLs: {duplicate_url_count}")

In [ ]:
df = pd.read_csv('boe_cbdc_articles.csv')
fig, ax1 = plt.subplots(figsize=(10, 6))
year_counts = df['year'].value_counts().sort_index()
ax1.barh(year_counts.index.astype(str), year_counts.values, color='#4a4a4a')
ax1.set_xlabel('Count', fontsize=12)
ax1.set_ylabel('Year', fontsize=12)
ax1.set_title('Publications by Year', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax2 = plt.subplots(figsize=(10, 6))
type_counts = df['type'].value_counts().sort_values()
ax2.barh(type_counts.index, type_counts.values, color='#4a4a4a')
ax2.set_xlabel('Count', fontsize=12)
ax2.set_ylabel('Type', fontsize=12)
ax2.set_title('Publications by Type', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import sent_tokenize
import nltk

In [ ]:
def clean_for_sentiment(text: str) -> str:

    text = str(text)
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^\w\s.,!?\'\"-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df = pd.read_csv('boe_cbdc_articles.csv')

df['content_sentiment'] = df['content_full'].apply(clean_for_sentiment)
df_sentiment = df[[
    'title', 'date', 'year', 'type', 'source', 'url',
    'content_full',
    'content_sentiment'
]].copy()

sentiment_output_path = "boe_articles_for_sentiment.csv"
df_sentiment.to_csv(sentiment_output_path, index=False, encoding="utf-8-sig")
print(f"SAVED: {sentiment_output_path}")
print(f"Columns: {df_sentiment.columns.tolist()}")
print(f"Rows: {len(df_sentiment)}")

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('words')
from nltk.corpus import words

In [ ]:
english_vocab = set(words.words())
base_stopwords = set(stopwords.words("english"))
custom_cbdc_stopwords = {
    "cbdc", "central", "bank", "banks", "currency",
    "financial", "monetary", "uk", "england", "pound"
}
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def clean_for_lda(text: str, use_lemmatization=True) -> str:

    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t in english_vocab or len(t) <= 3]
    text = " ".join(tokens)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in base_stopwords]
    tokens = [t for t in tokens if t not in {"nil", "na", "nan", "none"}]
    tokens = [t for t in tokens if t not in custom_cbdc_stopwords]
    tokens = [t for t in tokens if 3 <= len(t) <= 25]
    if use_lemmatization:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    else:
        tokens = [stemmer.stem(t) for t in tokens]
    text = " ".join(tokens).strip()

    return text

df = pd.read_csv('boe_cbdc_articles.csv')
df['content_lda_stem'] = df['content_full'].apply(
    lambda x: clean_for_lda(x, use_lemmatization=False)
)
df['content_lda_lemma'] = df['content_full'].apply(
    lambda x: clean_for_lda(x, use_lemmatization=True)
)

df_lda = df[[
    'title', 'date', 'year', 'type', 'source', 'url',
    'content_lda_stem',
    'content_lda_lemma'
]].copy()

lda_output_path = "boe_articles_for_lda.csv"
df_lda.to_csv(lda_output_path, index=False, encoding="utf-8-sig")
print(f"SAVED: {lda_output_path}")
print(f"Columns: {df_lda.columns.tolist()}")

In [ ]:
df = pd.read_csv('boe_articles_for_lda.csv')
all_text = ' '.join(df['content_lda_lemma'].dropna().astype(str))
words = all_text.split()

word_freq = Counter(words)
top_20 = word_freq.most_common(20)

words_list = [item[0] for item in top_20]
counts_list = [item[1] for item in top_20]

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(words_list[::-1], counts_list[::-1], color='dimgray', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Freq', fontsize=10)
ax.set_title('Top 20 Unigrams', fontsize=11, loc='center')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
df = pd.read_csv('boe_articles_for_lda.csv')
all_text = ' '.join(df['content_lda_lemma'].dropna().astype(str))
words = all_text.split()

word_bigrams = list(bigrams(words))
bigram_phrases = ['_'.join(pair) for pair in word_bigrams]

bigram_freq = Counter(bigram_phrases)
top_20_bigrams = bigram_freq.most_common(20)

bigrams_list = [item[0] for item in top_20_bigrams]
counts_list = [item[1] for item in top_20_bigrams]

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(bigrams_list[::-1], counts_list[::-1], color='dimgray', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Freq', fontsize=10)
ax.set_title('Top 20 Bigrams', fontsize=11, loc='center')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud

df = pd.read_csv('boe_articles_for_lda.csv')
all_text = ' '.join(df['content_lda_lemma'].dropna().astype(str))

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    max_words=150,
    relative_scaling=0.5,
    min_font_size=8,
    colormap='Blues',
    collocations=True
).generate(all_text)

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
df = pd.read_csv('boe_articles_for_lda.csv')
texts = df["content_lda_lemma"].fillna("").astype(str)

vectorizer = CountVectorizer(
    token_pattern=r"(?u)\b[a-z]{3,}\b",
    min_df=5,
    max_df=0.8,
)
dtm = vectorizer.fit_transform(texts)
terms = np.array(vectorizer.get_feature_names_out())

tfidf_transformer = TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)
dtm_tfidf = tfidf_transformer.fit_transform(dtm)

print(f"Documents: {dtm_tfidf.shape[0]}, Terms: {dtm_tfidf.shape[1]}")
print(f"Non-sparse entries: {dtm_tfidf.nnz}")
sparsity_before = 1 - (dtm_tfidf.nnz / (dtm_tfidf.shape[0] * dtm_tfidf.shape[1]))
print(f"Sparsity: {sparsity_before*100:.2f}%")
print(f"Maximal term length: {max(len(t) for t in terms)}")
print(f"Weighting: tf-idf")

terms_sample = terms[:10]
dtm_sample = dtm[:7, :10].toarray()
sample_df_before = pd.DataFrame(dtm_sample, columns=terms_sample, index=df.index[:7])
sample_df_before.index.name = "Docs"
print("Sample:")
print(sample_df_before.to_string())

GIBBS

In [ ]:
X = dtm

topics_range = list(range(2, 21))
gibbs_perplexity = []
gibbs_log_likelihood = []

for n_topics in topics_range:
    lda_gibbs = LatentDirichletAllocation(
        n_components=n_topics,
        learning_method='batch',
        random_state=1234,
        max_iter=1000,
        n_jobs=-1,
        verbose=0
    )

    lda_gibbs.fit(X)

    perplexity = lda_gibbs.perplexity(X)
    log_likelihood = lda_gibbs.score(X)

    gibbs_perplexity.append(perplexity)
    gibbs_log_likelihood.append(log_likelihood)

    print(f"Topics={n_topics:2d}: Perplexity={perplexity:8.2f}, Log-likelihood={log_likelihood:10.2f}")

perp_normalized = (np.array(gibbs_perplexity) - min(gibbs_perplexity)) / (max(gibbs_perplexity) - min(gibbs_perplexity))
loglik_normalized = (np.array(gibbs_log_likelihood) - min(gibbs_log_likelihood)) / (max(gibbs_log_likelihood) - min(gibbs_log_likelihood))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

ax1.plot(topics_range, perp_normalized, 'o-', color='black', linewidth=2, markersize=8)
ax1.set_ylabel('minimize', fontsize=10)
ax1.set_ylim(-0.1, 1.1)
ax1.grid(True, alpha=0.3, axis='x')
ax1.set_title('LDA tuning for Gibbs model', fontsize=12)

ax2.plot(topics_range, loglik_normalized, '^-', color='black', linewidth=2, markersize=8)
ax2.set_xlabel('number of topics', fontsize=10)
ax2.set_ylabel('maximize', fontsize=10)
ax2.set_ylim(-0.1, 1.1)
ax2.set_xticks(topics_range)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('lda_tuning_gibbs.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
np.random.seed(1234)

X = dtm
feature_names = vectorizer.get_feature_names_out()

k_range = range(14, 17)

all_models = {}
all_top_terms = {}

for k_gibbs in k_range:

    model_lda_gibbs = LatentDirichletAllocation(
        n_components=k_gibbs,
        learning_method='batch',
        random_state=1234,
        max_iter=1000,
        n_jobs=-1,
        verbose=0
    )

    model_lda_gibbs.fit(X)
    all_models[k_gibbs] = model_lda_gibbs

    terms_gibbs = []
    print(f"Top 5 terms for (k={k_gibbs}):")
    for topic_idx, topic in enumerate(model_lda_gibbs.components_):
        top_indices = topic.argsort()[-5:][::-1]
        top_terms = [feature_names[i] for i in top_indices]
        terms_gibbs.append(top_terms)
        print(f"  Topic {topic_idx + 1}: {', '.join(top_terms)}")

    all_top_terms[k_gibbs] = terms_gibbs

for k_gibbs in k_range:
    model = all_models[k_gibbs]

    n_cols = 4
    n_rows = int(np.ceil(k_gibbs / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 2.5))
    axes = axes.flatten()

    colors = plt.cm.Set3(np.linspace(0, 1, k_gibbs))

    for topic_idx in range(k_gibbs):
        top_indices = model.components_[topic_idx].argsort()[-5:][::-1]
        top_terms = [feature_names[i] for i in top_indices]
        top_betas = model.components_[topic_idx][top_indices]

        ax = axes[topic_idx]
        ax.barh(range(5), top_betas, color=colors[topic_idx],
                edgecolor='black', linewidth=0.5)
        ax.set_yticks(range(5))
        ax.set_yticklabels(top_terms[::-1], fontsize=7)
        ax.set_xlabel('Beta', fontsize=7)
        ax.set_title(f'{topic_idx + 1}', fontsize=9, fontweight='bold')
        ax.tick_params(axis='both', labelsize=7)
        ax.grid(True, alpha=0.3, axis='x')
        ax.invert_yaxis()

    for idx in range(k_gibbs, len(axes)):
        fig.delaxes(axes[idx])

    plt.suptitle(f'Top Terms for Gibbs Model (k={k_gibbs})',
                 fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

In [ ]:
np.random.seed(1234)
k_gibbs = 15

model_lda_gibbs = LatentDirichletAllocation(
    n_components=k_gibbs,
    learning_method='batch',
    random_state=1234,
    max_iter=1000,
    n_jobs=-1,
    verbose=0,
    batch_size=100
)
model_lda_gibbs.fit(dtm)

In [ ]:
feature_names = vectorizer.get_feature_names_out()
terms_gibbs = []

print("Top 5 Terms per Topic")
for topic_idx, topic in enumerate(model_lda_gibbs.components_):
    top_indices = topic.argsort()[-5:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    terms_gibbs.append(top_terms)
    print(f"Topic {topic_idx + 1}: {', '.join(top_terms)}")

beta_topics_gibbs = model_lda_gibbs.components_

In [ ]:
n_cols = 4
n_rows = int(np.ceil(k_gibbs / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 2.5))
axes = axes.flatten()
colors = plt.cm.Set3(np.linspace(0, 1, k_gibbs))

for topic_idx in range(k_gibbs):
    top_indices = model_lda_gibbs.components_[topic_idx].argsort()[-5:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    top_betas = model_lda_gibbs.components_[topic_idx][top_indices]

    ax = axes[topic_idx]
    ax.barh(range(5), top_betas, color=colors[topic_idx], edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(5))
    ax.set_yticklabels(top_terms[::-1], fontsize=7)
    ax.set_xlabel('Beta', fontsize=7)
    ax.set_title(f'{topic_idx + 1}', fontsize=9, fontweight='bold')
    ax.tick_params(axis='both', labelsize=7)
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()

for idx in range(k_gibbs, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle(f'Top Terms for Gibbs Model (k={k_gibbs})', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
gamma_gibbs = model_lda_gibbs.transform(dtm)
gammaDF_gibbs = pd.DataFrame(gamma_gibbs, columns=[f'Topic_{i+1}' for i in range(k_gibbs)])
gammaDF_gibbs.insert(0, 'doc_id', range(len(df)))

print(f"Gamma matrix shape: {gamma_gibbs.shape}")
print(gammaDF_gibbs.head())

In [ ]:
top_texts_by_topic = []

for topic_idx in range(k_gibbs):
    topic_col = f'Topic_{topic_idx + 1}'
    top_5_docs = gammaDF_gibbs.nlargest(5, topic_col)
    doc_ids = top_5_docs['doc_id'].tolist()
    gamma_values = top_5_docs[topic_col].tolist()
    texts = [df['content_lda_lemma'].iloc[doc_id] for doc_id in doc_ids]

    topic_df = pd.DataFrame({'doc_id': doc_ids, 'gamma_value': gamma_values, 'text': texts})
    top_texts_by_topic.append(topic_df)

    print(f"Topic {topic_idx + 1}:")
    print(f"Document IDs: {doc_ids}")
    print(f"Gamma values: {[f'{v:.4f}' for v in gamma_values]}")

VEM

In [ ]:
X = dtm
topics_range = list(range(2, 21))
vem_perplexity = []
vem_log_likelihood = []

for n_topics in topics_range:
    lda_vem = LatentDirichletAllocation(
        n_components=n_topics,
        learning_method='online',
        random_state=1234,
        max_iter=500,
        n_jobs=-1,
        verbose=0
    )

    lda_vem.fit(X)

    perplexity = lda_vem.perplexity(X)
    log_likelihood = lda_vem.score(X)

    vem_perplexity.append(perplexity)
    vem_log_likelihood.append(log_likelihood)

    print(f"Topics={n_topics:2d}: Perplexity={perplexity:8.2f}, Log-likelihood={log_likelihood:10.2f}")

perp_normalized = (np.array(vem_perplexity) - min(vem_perplexity)) / (max(vem_perplexity) - min(vem_perplexity))
loglik_normalized = (np.array(vem_log_likelihood) - min(vem_log_likelihood)) / (max(vem_log_likelihood) - min(vem_log_likelihood))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

ax1.plot(topics_range, perp_normalized, 'o-', color='black', linewidth=2, markersize=8)
ax1.set_ylabel('minimize', fontsize=10)
ax1.set_ylim(-0.1, 1.1)
ax1.grid(True, alpha=0.3, axis='x')
ax1.set_title('LDA tuning for VEM model', fontsize=12)

ax2.plot(topics_range, loglik_normalized, '^-', color='black', linewidth=2, markersize=8)
ax2.set_xlabel('number of topics', fontsize=10)
ax2.set_ylabel('maximize', fontsize=10)
ax2.set_ylim(-0.1, 1.1)
ax2.set_xticks(topics_range)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('lda_tuning_vem.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
np.random.seed(1234)

X = dtm
feature_names = vectorizer.get_feature_names_out()

k_range = range(7, 10)

all_models = {}
all_top_terms = {}

for k_vem in k_range:

    model_lda_vem = LatentDirichletAllocation(
        n_components=k_vem,
        learning_method='online',
        random_state=1234,
        max_iter=500,
        n_jobs=-1,
        verbose=0,
        batch_size=128,
        learning_offset=50.0,
        learning_decay=0.7
    )

    model_lda_vem.fit(X)
    all_models[k_vem] = model_lda_vem

    terms_vem = []
    print(f"Top 5 terms for (k={k_vem}):")
    for topic_idx, topic in enumerate(model_lda_vem.components_):
        top_indices = topic.argsort()[-5:][::-1]
        top_terms = [feature_names[i] for i in top_indices]
        terms_vem.append(top_terms)
        print(f"  Topic {topic_idx + 1}: {', '.join(top_terms)}")

    all_top_terms[k_vem] = terms_vem

for k_vem in k_range:
    model = all_models[k_vem]

    n_cols = 4
    n_rows = int(np.ceil(k_vem / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 2.5))
    axes = axes.flatten()

    colors = plt.cm.Set3(np.linspace(0, 1, k_vem))

    for topic_idx in range(k_vem):
        top_indices = model.components_[topic_idx].argsort()[-5:][::-1]
        top_terms = [feature_names[i] for i in top_indices]
        top_betas = model.components_[topic_idx][top_indices]

        ax = axes[topic_idx]
        ax.barh(range(5), top_betas, color=colors[topic_idx],
                edgecolor='black', linewidth=0.5)
        ax.set_yticks(range(5))
        ax.set_yticklabels(top_terms[::-1], fontsize=7)
        ax.set_xlabel('Beta', fontsize=7)
        ax.set_title(f'{topic_idx + 1}', fontsize=9, fontweight='bold')
        ax.tick_params(axis='both', labelsize=7)
        ax.grid(True, alpha=0.3, axis='x')
        ax.invert_yaxis()

    for idx in range(k_vem, len(axes)):
        fig.delaxes(axes[idx])

    plt.suptitle(f'Top Terms for VEM Model (k={k_vem})',
                 fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

In [ ]:
np.random.seed(1234)
k_vem = 8

model_lda_vem = LatentDirichletAllocation(
    n_components=k_vem,
    learning_method='online',
    random_state=1234,
    max_iter=500,
    n_jobs=-1,
    verbose=0,
    batch_size=128,
    learning_offset=50.0,
    learning_decay=0.7
)
model_lda_vem.fit(dtm)

In [ ]:
feature_names = vectorizer.get_feature_names_out()
terms_vem = []

print("Top 5 Terms per Topic")
for topic_idx, topic in enumerate(model_lda_vem.components_):
    top_indices = topic.argsort()[-5:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    terms_vem.append(top_terms)
    print(f"Topic {topic_idx + 1}: {', '.join(top_terms)}")

beta_topics_vem = model_lda_vem.components_

In [ ]:
n_cols = 4
n_rows = int(np.ceil(k_vem / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 2.5))
axes = axes.flatten()
colors = plt.cm.Set3(np.linspace(0, 1, k_vem))

for topic_idx in range(k_vem):
    top_indices = model_lda_vem.components_[topic_idx].argsort()[-5:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    top_betas = model_lda_vem.components_[topic_idx][top_indices]

    ax = axes[topic_idx]
    ax.barh(range(5), top_betas, color=colors[topic_idx], edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(5))
    ax.set_yticklabels(top_terms[::-1], fontsize=7)
    ax.set_xlabel('Beta', fontsize=7)
    ax.set_title(f'{topic_idx + 1}', fontsize=9, fontweight='bold')
    ax.tick_params(axis='both', labelsize=7)
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()

for idx in range(k_vem, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle(f'Top Terms for VEM Model (k={k_vem})', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
gamma_vem = model_lda_vem.transform(dtm)
gammaDF_vem = pd.DataFrame(gamma_vem, columns=[f'Topic_{i+1}' for i in range(k_vem)])
gammaDF_vem.insert(0, 'doc_id', range(len(df)))

print(f"Gamma matrix shape: {gamma_vem.shape}")
print(gammaDF_vem.head())

In [ ]:
top_texts_by_topic_vem = []

for topic_idx in range(k_vem):
    topic_col = f'Topic_{topic_idx + 1}'
    top_5_docs = gammaDF_vem.nlargest(5, topic_col)
    doc_ids = top_5_docs['doc_id'].tolist()
    gamma_values = top_5_docs[topic_col].tolist()
    texts = [df['content_lda_lemma'].iloc[doc_id] for doc_id in doc_ids]

    topic_df = pd.DataFrame({'doc_id': doc_ids, 'gamma_value': gamma_values, 'text': texts})
    top_texts_by_topic_vem.append(topic_df)

    print(f"Topic {topic_idx + 1}:")
    print(f"Document IDs: {doc_ids}")
    print(f"Gamma values: {[f'{v:.4f}' for v in gamma_values]}")

KMEANS CLUSTERING

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

svd = TruncatedSVD(n_components=50, random_state=1234)
X_svd = svd.fit_transform(dtm_tfidf)

print(f"Original shape: {dtm_tfidf.shape}")
print(f"Reduced shape: {X_svd.shape}")
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.2%}")

X = normalize(X_svd, norm='l2')
print(f"Normalised shape: {X.shape}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

X = normalize(svd.transform(dtm_tfidf), norm='l2')

k_range = range(2, 11)
inertias = []
silhouette_scores = []
calinski_scores = []
davies_bouldin_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=1234, n_init=100, max_iter=1000)
    labels = kmeans.fit_predict(X)

    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X, labels, metric="cosine"))
    calinski_scores.append(calinski_harabasz_score(X, labels))
    davies_bouldin_scores.append(davies_bouldin_score(X, labels))

optimal_k = k_range[np.argmax(silhouette_scores)]

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0, 0].set_ylabel('Inertia', fontsize=12)
axes[0, 0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xticks(k_range)

axes[0, 1].plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0, 1].set_ylabel('Silhouette Score', fontsize=12)
axes[0, 1].set_title('Silhouette Score', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xticks(k_range)

axes[1, 0].plot(k_range, calinski_scores, 'ro-', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1, 0].set_ylabel('Calinski-Harabasz Score', fontsize=12)
axes[1, 0].set_title('Calinski-Harabasz Score', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xticks(k_range)

axes[1, 1].plot(k_range, davies_bouldin_scores, 'mo-', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1, 1].set_ylabel('Davies-Bouldin Score', fontsize=12)
axes[1, 1].set_title('Davies-Bouldin Score', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticks(k_range)

plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter
from sklearn.preprocessing import normalize

X = normalize(svd.transform(dtm_tfidf), norm='l2')

k = 6
final_kmeans = KMeans(n_clusters=k, random_state=1234, n_init=100, max_iter=1000)
cluster_labels = final_kmeans.fit_predict(X)

df_original = pd.read_csv('boe_articles_for_lda.csv')
df_clustered = df_original.copy()
df_clustered['Cluster'] = cluster_labels
cluster_counts = Counter(cluster_labels)

print("Cluster Sizes")
for cluster_id in range(k):
    count = cluster_counts[cluster_id]
    pct = (count / len(cluster_labels)) * 100
    print(f"Cluster {cluster_id}: {count} articles ({pct:.1f}%)")

print("Top 10 Terms per Cluster")
X_raw = dtm_tfidf.toarray()
for cluster_id in range(k):
    cluster_mask = cluster_labels == cluster_id
    mean_term_freq = X_raw[cluster_mask].mean(axis=0)
    top_indices = mean_term_freq.argsort()[-10:][::-1]
    top_terms = [terms[i] for i in top_indices]
    print(f"Cluster {cluster_id} ({cluster_counts[cluster_id]} articles):")
    print(f" Top terms: {', '.join(top_terms)}")

HIERARCHICAL CLUSTERING

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

X = normalize(svd.transform(dtm_tfidf), norm='l2')

Z = linkage(X, method="ward")

k_values = [3, 4, 5]
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for idx, k_val in enumerate(k_values):
    cut_height = Z[-(k_val-1), 2]
    labels = fcluster(Z, t=k_val, criterion="maxclust")

    ax = axes[idx]
    dendrogram(Z, ax=ax, color_threshold=cut_height,
               above_threshold_color='black')
    ax.set_title(f'k = {k_val} clusters', fontsize=14, fontweight='bold')
    ax.set_xlabel('Sample Index', fontsize=10)
    ax.set_ylabel('Distance (Height)', fontsize=10)
    ax.axhline(y=cut_height, linestyle='--', color='red', linewidth=2,
               label=f'Cut at height {cut_height:.2f}')
    ax.legend()

plt.tight_layout()
plt.show()

for k_val in k_values:
    labels = fcluster(Z, t=k_val, criterion="maxclust")
    unique, counts = np.unique(labels, return_counts=True)
    print(f"\nk = {k_val}:")
    for cluster_id, count in zip(unique, counts):
        print(f"  Cluster {cluster_id}: {count} articles ({count/len(labels)*100:.1f}%)")

In [ ]:
k = 4
cut_height = Z[-(k-1), 2]

fig, ax = plt.subplots(figsize=(16, 6))

dendrogram(Z, ax=ax, color_threshold=cut_height,
           above_threshold_color='black')

ax.set_title(f'Hierarchical Clustering Dendrogram', fontsize=14, fontweight='bold')
ax.set_xlabel('Sample Index', fontsize=10)
ax.set_ylabel('Distance (Height)', fontsize=10)
ax.axhline(y=cut_height, linestyle='--', color='red', linewidth=2,
           label=f'Cut at height {cut_height:.2f}')
ax.legend()

labels = fcluster(Z, t=k, criterion="maxclust")
ddata = dendrogram(Z, ax=ax, color_threshold=cut_height,
                   above_threshold_color='black', no_plot=True)

leaves_order = ddata["leaves"]
labels_in_order = labels[leaves_order]
x_positions = np.arange(len(leaves_order)) * 10 + 5
colors = plt.cm.Set3(np.linspace(0, 1, k))
cluster_colors = {i+1: colors[i] for i in range(k)}

for cluster_id in np.unique(labels_in_order):
    idx = np.where(labels_in_order == cluster_id)[0]
    x_min = x_positions[idx].min() - 5
    x_max = x_positions[idx].max() + 5
    rect = plt.Rectangle(
        (x_min, 0), x_max - x_min, cut_height - 1e-6,
        fill=False, linewidth=2,
        edgecolor=cluster_colors[cluster_id]
    )
    ax.add_patch(rect)

plt.tight_layout()
plt.show()

In [ ]:
X_raw = dtm_tfidf.toarray()
cluster_counts_hier = Counter(labels)

print("Cluster Size")
for cluster_id in range(1, k+1):
    count = cluster_counts_hier[cluster_id]
    pct = (count / len(labels)) * 100
    print(f"Cluster {cluster_id}: {count} articles ({pct:.1f}%)")

print("\nTop 10 Terms per Cluster")
for cluster_id in range(1, k+1):  # fcluster labels start from 1
    cluster_mask = labels == cluster_id
    mean_term_freq = X_raw[cluster_mask].mean(axis=0)
    top_indices = mean_term_freq.argsort()[-10:][::-1]
    top_terms = [terms[i] for i in top_indices]
    print(f"Cluster {cluster_id} ({cluster_counts_hier[cluster_id]} articles):")
    print(f" Top terms: {', '.join(top_terms)}")

In [ ]:
k = 5
cut_height = Z[-(k-1), 2]

fig, ax = plt.subplots(figsize=(16, 6))

dendrogram(Z, ax=ax, color_threshold=cut_height,
           above_threshold_color='black')

ax.set_title(f'Hierarchical Clustering Dendrogram', fontsize=14, fontweight='bold')
ax.set_xlabel('Sample Index', fontsize=10)
ax.set_ylabel('Distance (Height)', fontsize=10)
ax.axhline(y=cut_height, linestyle='--', color='red', linewidth=2,
           label=f'Cut at height {cut_height:.2f}')
ax.legend()

labels = fcluster(Z, t=k, criterion="maxclust")
ddata = dendrogram(Z, ax=ax, color_threshold=cut_height,
                   above_threshold_color='black', no_plot=True)

leaves_order = ddata["leaves"]
labels_in_order = labels[leaves_order]
x_positions = np.arange(len(leaves_order)) * 10 + 5
colors = plt.cm.Set3(np.linspace(0, 1, k))
cluster_colors = {i+1: colors[i] for i in range(k)}

for cluster_id in np.unique(labels_in_order):
    idx = np.where(labels_in_order == cluster_id)[0]
    x_min = x_positions[idx].min() - 5
    x_max = x_positions[idx].max() + 5
    rect = plt.Rectangle(
        (x_min, 0), x_max - x_min, cut_height - 1e-6,
        fill=False, linewidth=2,
        edgecolor=cluster_colors[cluster_id]
    )
    ax.add_patch(rect)

plt.tight_layout()
plt.show()

In [ ]:
X_raw = dtm_tfidf.toarray()
cluster_counts_hier = Counter(labels)

print("Cluster Size")
for cluster_id in range(1, k+1):
    count = cluster_counts_hier[cluster_id]
    pct = (count / len(labels)) * 100
    print(f"Cluster {cluster_id}: {count} articles ({pct:.1f}%)")

print("Top 10 Terms per Cluster")
for cluster_id in range(1, k+1):  # fcluster labels start from 1
    cluster_mask = labels == cluster_id
    mean_term_freq = X_raw[cluster_mask].mean(axis=0)
    top_indices = mean_term_freq.argsort()[-10:][::-1]
    top_terms = [terms[i] for i in top_indices]
    print(f"Cluster {cluster_id} ({cluster_counts_hier[cluster_id]} articles):")
    print(f" Top terms: {', '.join(top_terms)}")

In [ ]:
from transformers import pipeline

pipe = pipeline("text-classification", model="ProsusAI/finbert")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

In [ ]:
import seaborn as sns
import torch
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# https://huggingface.co/ProsusAI/finbert
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
finbert = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

In [ ]:
nltk.download("punkt")
nltk.download("punkt_tab")

FINBERT

In [ ]:
df = pd.read_csv('boe_articles_for_sentiment.csv')
text_column = 'content_sentiment'

if 'year' in df.columns:
    df['Year'] = df['year']
elif 'date' in df.columns:
    df['Year'] = pd.to_datetime(df['date'], errors='coerce').dt.year

finbert = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=-1
)

In [ ]:
def analyze_finbert(text, max_length=512):
    if pd.isna(text) or str(text).strip() == "":
        return {'label': 'NEUTRAL', 'score': 0.0}
    text = str(text)[:max_length * 4]
    try:
        result = finbert(text, truncation=True, max_length=max_length)[0]
        return {'label': result['label'].upper(), 'score': result['score']}
    except:
        return {'label': 'NEUTRAL', 'score': 0.0}

finbert_results = []
for text in tqdm(df[text_column], desc="Progress", total=len(df)):
    finbert_results.append(analyze_finbert(text))

df['FinBERT_Sentiment'] = [r['label'] for r in finbert_results]
df['FinBERT_Score'] = [r['score'] for r in finbert_results]

def convert_to_sentiment_score(label, confidence):
    if label == 'POSITIVE':
        return confidence
    elif label == 'NEGATIVE':
        return -confidence
    else:
        return 0.0

df['FinBERT_Sentiment_Score'] = df.apply(
    lambda row: convert_to_sentiment_score(row['FinBERT_Sentiment'], row['FinBERT_Score']),
    axis=1
)

In [ ]:
counts = df['FinBERT_Sentiment'].value_counts()
for sentiment in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
    count = counts.get(sentiment, 0)
    pct = count / len(df) * 100
    print(f"{sentiment:10s}: {count:4d} articles ({pct:5.1f}%)")

print(f"Mean:   {df['FinBERT_Sentiment_Score'].mean():.3f}")
print(f"Median: {df['FinBERT_Sentiment_Score'].median():.3f}")
print(f"Std deviation:{df['FinBERT_Sentiment_Score'].std():.3f}")

In [ ]:
sns.set_style("whitegrid")
colors = {'POSITIVE': '#22c55e', 'NEGATIVE': '#ef4444', 'NEUTRAL': '#94a3b8'}

has_year = 'Year' in df.columns and df['Year'].notna().sum() > 0

if has_year:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
else:
    fig, axes = plt.subplots(1, 1, figsize=(7, 5))
    axes = [axes]

ax = axes[0]
ax.hist(df['FinBERT_Sentiment_Score'], bins=30, edgecolor='black', alpha=0.7, color='#3b82f6')
ax.axvline(0, color='gray', linestyle='--', linewidth=2, label='Neutral (0)')
ax.axvline(0.1, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Positive threshold')
ax.axvline(-0.1, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Negative threshold')
ax.set_xlabel('Sentiment Score', fontsize=11)
ax.set_ylabel('Number of Documents', fontsize=11)
ax.set_title('FinBERT Sentiment Score Distribution', fontsize=13, fontweight='bold')
ax.set_ylim(0, 160)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

if has_year:
    ax = axes[1]
    yearly_pct = pd.crosstab(df['Year'], df['FinBERT_Sentiment'], normalize='index') * 100
    for sentiment in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
        if sentiment in yearly_pct.columns:
            ax.plot(yearly_pct.index, yearly_pct[sentiment],
                    marker='o', linewidth=2, markersize=8,
                    color=colors.get(sentiment, '#94a3b8'), label=sentiment)
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Percentage (%)', fontsize=11)
    ax.set_title('Sentiment Trend Over Time', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
df.to_csv('finbert_results.csv', index=False, encoding='utf-8-sig')

LEXICON BASED

In [ ]:
CBDC_POSITIVE_WORDS = {
    'innovation', 'innovative', 'technological', 'advancement', 'progress',
    'benefit', 'benefits', 'advantage', 'advantages', 'opportunity',
    'efficiency', 'efficient', 'effective', 'improvement', 'improved',
    'enhance', 'enhanced', 'optimize', 'optimized',
    'inclusion', 'inclusive', 'accessibility', 'accessible',
    'secure', 'security', 'trust', 'trustworthy', 'reliable', 'stability',
    'resilient', 'resilience', 'robust', 'safe', 'safety',
    'adoption', 'success', 'successful', 'growth', 'develop', 'development',
    'affordable', 'savings', 'streamline', 'convenient', 'faster',
    'promising', 'potential', 'enable', 'facilitate', 'support', 'strengthen'
}

CBDC_NEGATIVE_WORDS = {
    'risk', 'risks', 'risky', 'concern', 'concerns', 'worried', 'worry',
    'threat', 'threats', 'danger', 'dangerous', 'hazard',
    'privacy', 'surveillance', 'monitoring', 'tracking', 'intrusive',
    'breach', 'violation', 'abuse', 'misuse',
    'vulnerability', 'vulnerable', 'hack', 'hacking', 'fraud', 'criminal',
    'instability', 'unstable', 'disrupt', 'disruption', 'crisis', 'collapse',
    'challenge', 'challenges', 'difficult', 'difficulty',
    'barrier', 'obstacle', 'complexity', 'complex',
    'costly', 'expensive', 'burden', 'burdensome',
    'fail', 'failure', 'problem', 'problems', 'issue', 'issues',
    'flaw', 'limitation', 'limitations',
    'opposition', 'resist', 'resistance', 'reject', 'rejection',
    'skeptical', 'skepticism', 'doubt', 'uncertain', 'uncertainty',
    'undermine', 'weaken', 'hinder', 'prevent', 'adverse', 'harmful', 'damage'
}

In [ ]:
df = pd.read_csv('boe_articles_for_sentiment.csv')
text_column = 'content_sentiment'

def analyze_cbdc_sentiment(text, threshold=0.1):
    if pd.isna(text) or str(text).strip() == "":
        return {'label': 'NEUTRAL', 'pos_count': 0, 'neg_count': 0, 'score': 0.0}
    text_lower = str(text).lower()
    pos_count = sum(1 for word in CBDC_POSITIVE_WORDS if word in text_lower)
    neg_count = sum(1 for word in CBDC_NEGATIVE_WORDS if word in text_lower)
    total = pos_count + neg_count
    if total == 0:
        return {'label': 'NEUTRAL', 'pos_count': 0, 'neg_count': 0, 'score': 0.0}
    score = (pos_count - neg_count) / total
    label = 'POSITIVE' if score > threshold else 'NEGATIVE' if score < -threshold else 'NEUTRAL'
    return {'label': label, 'pos_count': pos_count, 'neg_count': neg_count, 'score': score}

results = [analyze_cbdc_sentiment(text) for text in df[text_column]]
df['Lexicon_Sentiment'] = [r['label'] for r in results]
df['Lexicon_Positive_Words'] = [r['pos_count'] for r in results]
df['Lexicon_Negative_Words'] = [r['neg_count'] for r in results]
df['Lexicon_Score'] = [r['score'] for r in results]

In [ ]:
sentiment_counts = df['Lexicon_Sentiment'].value_counts()

for sentiment in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    count = sentiment_counts.get(sentiment, 0)
    pct = count / len(df) * 100
    print(f"{sentiment:10s}: {count:4d} documents ({pct:5.1f}%)")

print(f"Avg positive words: {df['Lexicon_Positive_Words'].mean():.2f}")
print(f"Avg negative words: {df['Lexicon_Negative_Words'].mean():.2f}")
print(f"Mean score:         {df['Lexicon_Score'].mean():.3f}")
print(f"Median score:       {df['Lexicon_Score'].median():.3f}")
print(f"Std deviation:      {df['Lexicon_Score'].std():.3f}")

In [ ]:
colors = {'POSITIVE': '#22c55e', 'NEGATIVE': '#ef4444', 'NEUTRAL': '#94a3b8'}
has_year = 'year' in df.columns and df['year'].notna().sum() > 0

if has_year:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
else:
    fig, axes = plt.subplots(1, 1, figsize=(7, 5))
    axes = [axes]

ax = axes[0]
ax.hist(df['Lexicon_Score'], bins=30, edgecolor='black', alpha=0.7, color='#3b82f6')
ax.axvline(0, color='gray', linestyle='--', linewidth=2, label='Neutral (0)')
ax.axvline(0.1, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Positive threshold')
ax.axvline(-0.1, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Negative threshold')
ax.set_xlabel('Sentiment Score', fontsize=11)
ax.set_ylabel('Number of Documents', fontsize=11)
ax.set_title('Custom Lexicon Sentiment Score Distribution', fontsize=13, fontweight='bold')
ax.set_ylim(0, 30)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

if has_year:
    ax = axes[1]
    yearly_pct = pd.crosstab(df['year'], df['Lexicon_Sentiment'], normalize='index') * 100
    for sentiment in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
        if sentiment in yearly_pct.columns:
            ax.plot(yearly_pct.index, yearly_pct[sentiment],
                    marker='o', linewidth=2, markersize=8,
                    color=colors.get(sentiment, '#94a3b8'), label=sentiment)
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Percentage (%)', fontsize=11)
    ax.set_title('Sentiment Trend Over Time', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(df['Lexicon_Score'], bins=30, edgecolor='black', alpha=0.7, color='#3b82f6')
ax.axvline(0, color='gray', linestyle='--', linewidth=2, label='Neutral (0)')
ax.axvline(0.1, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Positive threshold')
ax.axvline(-0.1, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Negative threshold')
ax.set_xlabel('Sentiment Score', fontsize=11)
ax.set_ylabel('Number of Documents', fontsize=11)
ax.set_title('Custom Lexicon Sentiment Score Distribution', fontsize=13, fontweight='bold')
ax.set_ylim(0, 160)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
df.to_csv('cbdc_lexicon_sentiment_results.csv', index=False, encoding='utf-8')

HYBRID: COMBINED BOTH

In [ ]:
df_finbert = pd.read_csv('finbert_results.csv')
df_lexicon = pd.read_csv('cbdc_lexicon_sentiment_results.csv')

df = df_finbert.copy()
df['Lexicon_Sentiment'] = df_lexicon['Lexicon_Sentiment']
df['Lexicon_Positive_Words'] = df_lexicon['Lexicon_Positive_Words']
df['Lexicon_Negative_Words'] = df_lexicon['Lexicon_Negative_Words']
df['Lexicon_Score'] = df_lexicon['Lexicon_Score']

if 'year' in df.columns:
    df['Year'] = df['year']
elif 'date' in df.columns:
    df['Year'] = pd.to_datetime(df['date'], errors='coerce').dt.year

print(f"FinBERT: {len(df_finbert)} articles")
print(f"Lexicon: {len(df_lexicon)} articles")
print(f"Combined: {len(df)} articles")

In [ ]:
def combine_sentiments(finbert_label, finbert_score, lexicon_label, lexicon_total):
    if finbert_label == lexicon_label:
        return finbert_label
    if lexicon_total >= 5:
        return lexicon_label
    if finbert_score > 0.8:
        return finbert_label
    return 'NEUTRAL'

def create_hybrid_score(row):
    finbert_score = row.get('FinBERT_Sentiment_Score', 0)
    lexicon_score = row['Lexicon_Score']
    if row['FinBERT_Sentiment'] == row['Lexicon_Sentiment']:
        return (finbert_score + lexicon_score) / 2
    if row['Lexicon_Total'] >= 5:
        return lexicon_score
    if row['FinBERT_Score'] > 0.8:
        return finbert_score
    return (finbert_score + lexicon_score) / 2

df['Lexicon_Total'] = df['Lexicon_Positive_Words'] + df['Lexicon_Negative_Words']
df['Hybrid_Sentiment'] = df.apply(
    lambda row: combine_sentiments(row['FinBERT_Sentiment'], row['FinBERT_Score'],
                                   row['Lexicon_Sentiment'], row['Lexicon_Total']), axis=1)
df['Hybrid_Score'] = df.apply(create_hybrid_score, axis=1)

In [ ]:
counts = df['Hybrid_Sentiment'].value_counts()
for sentiment in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
    count = counts.get(sentiment, 0)
    pct = count / len(df) * 100
    print(f"{sentiment:10s}: {count:4d} articles ({pct:5.1f}%)")

agree = (df['FinBERT_Sentiment'] == df['Lexicon_Sentiment']).sum()
print(f"FinBERT-Lexicon Agreement: {agree}/{len(df)} ({agree/len(df)*100:.1f}%)")
print(f"Mean score:    {df['Hybrid_Score'].mean():.3f}")
print(f"Median score:  {df['Hybrid_Score'].median():.3f}")
print(f"Std deviation: {df['Hybrid_Score'].std():.3f}")

In [ ]:
colors = {'POSITIVE': '#22c55e', 'NEGATIVE': '#ef4444', 'NEUTRAL': '#94a3b8'}
has_year = 'Year' in df.columns and df['Year'].notna().sum() > 0

if has_year:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
else:
    fig, axes = plt.subplots(1, 1, figsize=(7, 5))
    axes = [axes]

ax = axes[0]
ax.hist(df['Hybrid_Score'], bins=30, edgecolor='black', alpha=0.7, color='#3b82f6')
ax.axvline(0, color='gray', linestyle='--', linewidth=2, label='Neutral (0)')
ax.axvline(0.1, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Positive threshold')
ax.axvline(-0.1, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Negative threshold')
ax.set_xlabel('Sentiment Score', fontsize=11)
ax.set_ylabel('Number of Documents', fontsize=11)
ax.set_title('Hybrid Sentiment Score Distribution', fontsize=13, fontweight='bold')
ax.set_ylim(0, 30)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

if has_year:
    ax = axes[1]
    yearly_pct = pd.crosstab(df['Year'], df['Hybrid_Sentiment'], normalize='index') * 100
    for sentiment in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
        if sentiment in yearly_pct.columns:
            ax.plot(yearly_pct.index, yearly_pct[sentiment],
                    marker='o', linewidth=2, markersize=8,
                    color=colors.get(sentiment, '#94a3b8'), label=sentiment)
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Percentage (%)', fontsize=11)
    ax.set_title('Sentiment Trend Over Time', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(df['Hybrid_Score'], bins=30, edgecolor='black', alpha=0.7, color='#3b82f6')
ax.axvline(0, color='gray', linestyle='--', linewidth=2, label='Neutral (0)')
ax.axvline(0.1, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Positive threshold')
ax.axvline(-0.1, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Negative threshold')
ax.set_xlabel('Sentiment Score', fontsize=11)
ax.set_ylabel('Number of Documents', fontsize=11)
ax.set_title('Hybrid Sentiment Score Distribution', fontsize=13, fontweight='bold')
ax.set_ylim(0, 160)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
df.to_csv('boe_cbdc_sentiment_analysis.csv', index=False, encoding='utf-8-sig')

COMPARING

In [ ]:
methods = {
    'FinBERT': 'FinBERT_Sentiment',
    'CBDC Lexicon': 'Lexicon_Sentiment',
    'Hybrid': 'Hybrid_Sentiment'
}

comparison_data = pd.DataFrame({
    method: df[col].value_counts() for method, col in methods.items()
}).T.fillna(0)
print(comparison_data)

colors = {'POSITIVE': '#22c55e', 'NEGATIVE': '#ef4444', 'NEUTRAL': '#94a3b8'}
fig, ax = plt.subplots(figsize=(10, 6))
comparison_data.plot(
    kind='bar',
    ax=ax,
    color=[colors.get(c, '#94a3b8') for c in comparison_data.columns],
    edgecolor='black',
    linewidth=1
)

ax.set_title('Sentiment Distribution Comparison Across Methods',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Method', fontsize=12)
ax.set_ylabel('Number of Articles', fontsize=12)
ax.legend(title='Sentiment', loc='upper right')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()